#### <span style="color:blue">Azul:</span> Aluno 1

#### <span style="color:red">Vermelho:</span> Gabriel

#### <span style="color:green">Verde:</span> Wendell

#### <span style="color:red">Gabriel</span>

#### Passos

1. **Pré-processamento**
- Agrupar dados em janelas de 30s (as classificações pelos médicos são feitas analisando os ultimos 30s, então podemos agrupar por blocos de 30s para simplificar a quantidade de dados)
- Reduzir quantidade de dados e padronizar análise

2. **Gráficos**
- Duração total do sono e fases
- Duração de cada fase
- Interrupções do sono
- Qualidade do sono
- Análise por vários dias / semanal
- Relação entre atividade do EEG e fases

3. **Validação**
- Treinar com EDF classificado (T1)
- Gerar gráficos de saída
- Aplicar em EDF sem classificação (T2)
- Comparar resultados

4. **Importante**
- Não usar os mesmos EDFs em treino e teste

In [92]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mne
from scipy.signal import welch
from scipy.stats import skew, kurtosis
from scipy.integrate import trapezoid

In [93]:
#2.1 Carregamento dos dados
dados_dir = Path("../dados/edfs_originais")
saida_dir = Path("../dados/brutos")
dados_dir

WindowsPath('../dados/edfs_originais')

In [94]:
#3 Leitura e organização
#função para extrair cod do edf para procurar o hypnograma correspondente
def peganome(filename: str) -> str:
    base = Path(filename).name.split("-")[0]
    return base[:7]

def encontrapar(data_dir: Path, prefix="SC"): #procura por arquivos que comecem com SC
    arquivospsg = sorted(data_dir.glob(f"{prefix}*-PSG.edf")) #procura por aquivos que comecem com SC e terminem com -PSG.edf
    arquivoshyp = sorted(data_dir.glob(f"{prefix}*-Hypnogram.edf"))
    dicpsg = {peganome(arquivo.name): arquivo for arquivo in arquivospsg}
    dichyp = {peganome(arquivo.name): arquivo for arquivo in arquivoshyp}
    dicjunto = sorted(set(dicpsg) & set(dichyp)) #encontra os arquivos que tem PSG e Hypnograma correspondentes
    return [(k, dicpsg[k], dichyp[k]) for k in dicjunto]

pares = encontrapar(dados_dir, prefix="SC") #ordena as cahves e retorna tupla de pares
len(pares), pares[:5]

(1,
 [('SC4001E',
   WindowsPath('../dados/edfs_originais/SC4001E0-PSG.edf'),
   WindowsPath('../dados/edfs_originais/SC4001EC-Hypnogram.edf'))])

In [95]:
# Escolher quantidade de noites pra analisar
totalpares = 10
selecpares = pares[:totalpares]
selecpares

[('SC4001E',
  WindowsPath('../dados/edfs_originais/SC4001E0-PSG.edf'),
  WindowsPath('../dados/edfs_originais/SC4001EC-Hypnogram.edf'))]

In [96]:
# Conferir se par foi carregado corretamente
chavepar, caminhopsg, caminhohyp = selecpares[0]

bruto = mne.io.read_raw_edf(caminhopsg, preload=False, infer_types=True, verbose="ERROR")
anot = mne.read_annotations(caminhohyp)

print(f"Chave do par: {chavepar}")
print(f"PSG: {caminhopsg}")
print(f"Hypnograma: {caminhohyp}")
print(f"Canais: {bruto.ch_names}")
print(f"Frequência de amostragem: {bruto.info['sfreq']} Hz")
print("\nPrimeiras anotações:")
print(pd.DataFrame({
    "Início(s)": anot.onset[:10],
    "Duração(s)": anot.duration[:10],
    "Classificação": anot.description[:10]
}))

Chave do par: SC4001E
PSG: ..\dados\edfs_originais\SC4001E0-PSG.edf
Hypnograma: ..\dados\edfs_originais\SC4001EC-Hypnogram.edf
Canais: ['Fpz-Cz', 'Pz-Oz', 'horizontal', 'oro-nasal', 'submental', 'rectal', 'Event marker']
Frequência de amostragem: 100.0 Hz

Primeiras anotações:
   Início(s)  Duração(s)  Classificação
0        0.0     30630.0  Sleep stage W
1    30630.0       120.0  Sleep stage 1
2    30750.0       390.0  Sleep stage 2
3    31140.0        30.0  Sleep stage 3
4    31170.0        30.0  Sleep stage 2
5    31200.0       150.0  Sleep stage 3
6    31350.0        30.0  Sleep stage 4
7    31380.0        60.0  Sleep stage 3
8    31440.0        60.0  Sleep stage 4
9    31500.0        30.0  Sleep stage 3


In [97]:
#4 Juntar em blocos de 30s, juntar estagio 3 e 4 (recomendação mne), selecionar canal principal, juntar com anotações, remover longas partes do acordado, extrair dados
AnotacaoParaId = {
    "Sleep stage W": 1,
    "Sleep stage 1": 2,
    "Sleep stage 2": 3,
    "Sleep stage 3": 4, #juntar estagio 3 e 4
    "Sleep stage 4": 4, #juntar estagio 3 e 4
    "Sleep stage R": 5,
}

IdPraClassificacao = {
    1: "Acordado",
    2: "N1",
    3: "N2",
    4: "N3",
    5: "REM"
}

def escolher_canal(bruto, escolha="Fpz-Cz"):
    if escolha in bruto.ch_names:
        return escolha
    else:
        raise ValueError(f"Canal {escolha} não encontrado. Canais disponíveis: {bruto.ch_names}")
    
def extrair_dados_epocas(x, fs):
    n_fft = min(len(x), int(4*fs))  # Tamanho da janela de FFT (4 segundos ou o tamanho do segmento, o que for menor)
    delta = mne.time_frequency.band_power(x, sfreq=fs, fmin=0.5, fmax=4, method='welch', n_fft=n_fft)
    theta = mne.time_frequency.band_power(x, sfreq=fs, fmin=4, fmax=8, method='welch', n_fft=n_fft)
    alpha = mne.time_frequency.band_power(x, sfreq=fs, fmin=8, fmax=13, method='welch', n_fft=n_fft)
    beta = mne.time_frequency.band_power(x, sfreq=fs, fmin=13, fmax=30, method='welch', n_fft=n_fft)
    total = mne.time_frequency.band_power(x, sfreq=fs, fmin=0.5, fmax=30, method='welch', n_fft=n_fft)
    eps = 1e-12
    return {
        "média": np.mean(x),
        "desvio_padrão": np.std(x, ddof=1),
        "variância": np.var(x, ddof=1),
        "mínimo": np.min(x),
        "máximo": np.max(x),
        "pico_a_pico": np.ptp(x),
        "valor_rms": np.sqrt(np.mean(x**2)),
        "assimetria": skew(x, bias=False),
        "curtose_excesso": kurtosis(x, fisher=True, bias=False),
        "potência_delta": delta,
        "potência_theta": theta,
        "potência_alpha": alpha,
        "potência_beta": beta,
        "potência_total": total,
        "relativo_delta": delta / (total + eps),
        "relativo_theta": theta / (total + eps),
        "relativo_alpha": alpha / (total + eps),
        "relativo_beta": beta / (total + eps),
        "razão_delta_theta": delta / (theta + eps),
        "razão_delta_alpha": delta / (alpha + eps),
    }

Falta
- Gerar as epocas
- Pegar as epocas passar os dados pra função extrair_dados_epocas
- Passar pra CSV pra exibir

#### <span style="color:green">Wendell




In [98]:
def lim_sono(anot, margem_segundos=1800):
    ind_sono = np.where(anot.description != "Sleep stage W")[0]

    primeiro_ind = ind_sono[0]
    ultimo_ind = ind_sono[-1]

    ini_recorte = max(0, anot.onset[primeiro_ind] - margem_segundos)
    fim_recorte = anot.onset[ultimo_ind] + anot.duration[ultimo_ind] + margem_segundos

    return ini_recorte, fim_recorte

In [99]:
ini_recorte, fim_recorte = lim_sono(anot)
tempo_maximo_edf = bruto.times[-1]
fim_recorte = min(fim_recorte, tempo_maximo_edf)

bruto_recortado = bruto.copy().crop(tmin=ini_recorte, tmax=fim_recorte)

canal_alvo = escolher_canal(bruto_recortado)

events, event_id = mne.events_from_annotations(
    bruto_recortado,
    event_id=AnotacaoParaId,
    chunk_duration=30.0
)

tmax = 29.99
epocas = mne.Epochs(
    bruto_recortado,
    events,
    event_id=event_id,
    tmin=0,
    tmax=tmax,
    picks=[canal_alvo],
    baseline=None,
    preload=True
)

print(f"Total de épocas geradas: {len(epocas)}")


ValueError: No matching events found for Sleep stage W (event id 1)